In [ ]:
import os
import cv2
import numpy as np
import torch
from PIL import Image
from transformers import Sam3Processor, Sam3Model


# =========================
# CONFIG
# =========================

VIDEO_PATH      = r"G:\2026-Research-Project\admision_test\2026-04-15 11-00-45.mp4"
START_FRAME     = 0
MAX_FRAMES      = -1                    # -1 = toàn bộ, process ALL frames

# --- SAM3 ---
SAM3_MODEL_ID   = "facebook/sam3"
SAM3_TEXT_PROMPT = "vehicle"
SAM3_THRESH     = 0.5
SAM3_MASK_THRESH = 0.5
SAM3_BOX_SCORE_THR = 0.5
SAM3_DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
SAM3_USE_AMP    = (SAM3_DEVICE == "cuda")

# --- Mask overlay ---
MASK_COLOR      = (255, 0, 0)          # Xanh dương BGR
MASK_ALPHA      = 0.5
CONTOUR_THICK   = 2

# --- Output video ---
OUTPUT_DIR      = r"G:\2026-Research-Project\admision_test/output_sam3_masks"
OUTPUT_VIDEO    = os.path.join(OUTPUT_DIR, "segmented_output.mp4")

# --- Vùng loại trừ ---
EXCLUDE_ZONES = [
    [1040, 260, 1150, 360],
]
EXCLUDE_ZONE_MARGIN = 30


os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# SAM3 DETECT WITH MASKS
# =========================
def sam3_detect_masks(img_bgr, text_prompt):
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(img_rgb)
    H, W = img_bgr.shape[:2]

    inputs = sam3_processor(images=pil, text=text_prompt, return_tensors="pt")
    inputs = {k: v.to(SAM3_DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        if SAM3_USE_AMP:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = sam3_model(**inputs)
        else:
            outputs = sam3_model(**inputs)

    results = sam3_processor.post_process_instance_segmentation(
        outputs,
        threshold=SAM3_THRESH,
        mask_threshold=SAM3_MASK_THRESH,
        target_sizes=inputs["original_sizes"].tolist()
    )[0]

    # --- Cách 1: field "masks" trực tiếp ---
    masks_tensor = results.get("masks", None)
    scores = results.get("scores", None)
    boxes = results.get("boxes", None)

    if masks_tensor is not None and len(masks_tensor) > 0:
        detected = []
        scores_np = scores.detach().cpu().numpy().astype(np.float32) if scores is not None else np.ones(len(masks_tensor), dtype=np.float32)
        masks_np = masks_tensor.detach().cpu().numpy()

        for i in range(len(masks_np)):
            if scores_np[i] < SAM3_BOX_SCORE_THR:
                continue
            mask = masks_np[i]
            if mask.ndim == 3:
                mask = mask.squeeze(0)
            mask_bool = mask > 0.5

            if _in_exclude_zone(mask_bool):
                continue
            detected.append(mask_bool)
        return detected

    # --- Cách 2: segmentation map ---
    seg_map = results.get("segmentation", None)
    segments_info = results.get("segments_info", [])

    if seg_map is not None and len(segments_info) > 0:
        seg_np = seg_map.cpu().numpy()
        detected = []
        for seg in segments_info:
            score = seg.get("score", 1.0)
            if score < SAM3_BOX_SCORE_THR:
                continue
            mask = (seg_np == seg["id"])
            if _in_exclude_zone(mask):
                continue
            detected.append(mask)
        return detected

    # --- Cách 3: fallback từ boxes ---
    if boxes is not None and len(boxes) > 0:
        boxes_np = np.round(boxes.detach().cpu().numpy()).astype(np.int32)
        scores_np = scores.detach().cpu().numpy().astype(np.float32) if scores is not None else np.ones(len(boxes_np), dtype=np.float32)
        detected = []
        for i in range(len(boxes_np)):
            if scores_np[i] < SAM3_BOX_SCORE_THR:
                continue
            x1, y1, x2, y2 = boxes_np[i]
            mask = np.zeros((H, W), dtype=bool)
            mask[max(0, y1):min(H, y2), max(0, x1):min(W, x2)] = True
            if _in_exclude_zone(mask):
                continue
            detected.append(mask)
        return detected

    return []


def _in_exclude_zone(mask_bool):
    if not EXCLUDE_ZONES or not mask_bool.any():
        return False
    ys, xs = np.where(mask_bool)
    cx, cy = xs.mean(), ys.mean()
    for zone in EXCLUDE_ZONES:
        zx1 = zone[0] - EXCLUDE_ZONE_MARGIN
        zy1 = zone[1] - EXCLUDE_ZONE_MARGIN
        zx2 = zone[2] + EXCLUDE_ZONE_MARGIN
        zy2 = zone[3] + EXCLUDE_ZONE_MARGIN
        if zx1 <= cx <= zx2 and zy1 <= cy <= zy2:
            return True
    return False


def draw_masks_on_frame(img_bgr, masks):
    """Vẽ tất cả masks bằng 1 màu xanh dương duy nhất."""
    if not masks:
        return img_bgr

    # Gộp tất cả masks thành 1 combined mask
    combined = np.zeros(img_bgr.shape[:2], dtype=bool)
    for m in masks:
        combined |= m

    # Overlay màu xanh dương
    overlay = img_bgr.copy()
    overlay[combined] = MASK_COLOR

    result = cv2.addWeighted(overlay, MASK_ALPHA, img_bgr, 1 - MASK_ALPHA, 0)

    # Vẽ contour
    contour_mask = combined.astype(np.uint8) * 255
    contours, _ = cv2.findContours(contour_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(result, contours, -1, MASK_COLOR, CONTOUR_THICK, cv2.LINE_AA)

    return result


# =========================
# LOAD MODEL
# =========================
print(f"[INFO] Loading SAM3: {SAM3_MODEL_ID} | device: {SAM3_DEVICE}")
sam3_processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID)
sam3_model = Sam3Model.from_pretrained(SAM3_MODEL_ID).to(SAM3_DEVICE)
sam3_model.eval()
print("[INFO] SAM3 loaded!")


# =========================
# RUN ON VIDEO → OUTPUT VIDEO
# =========================
if not os.path.isfile(VIDEO_PATH):
    raise RuntimeError(f"Video not found: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
W_vid = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_vid = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Video writer — mp4v codec
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(OUTPUT_VIDEO, fourcc, fps, (W_vid, H_vid))

if not writer.isOpened():
    raise RuntimeError(f"Cannot create output video: {OUTPUT_VIDEO}")

print("=" * 60)
print("  SAM3 VIDEO MASK → VIDEO OUTPUT")
print("=" * 60)
print(f"  Input:       {VIDEO_PATH}")
print(f"  Resolution:  {W_vid}x{H_vid} @ {fps:.1f} FPS")
print(f"  Total:       {total} frames")
print(f"  Prompt:      '{SAM3_TEXT_PROMPT}'")
print(f"  Threshold:   {SAM3_THRESH}")
print(f"  Device:      {SAM3_DEVICE}")
print(f"  Mask color:  Blue (BGR {MASK_COLOR})")
print(f"  Output:      {OUTPUT_VIDEO}")
print("=" * 60)

if START_FRAME > 0:
    cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

frame_idx = START_FRAME
processed = 0
total_dets = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if 0 < MAX_FRAMES <= processed:
        break

    # Detect masks cho EVERY frame
    masks = sam3_detect_masks(frame, SAM3_TEXT_PROMPT)

    # Vẽ mask overlay
    result = draw_masks_on_frame(frame, masks)

    # Ghi vào video output
    writer.write(result)

    n = len(masks)
    total_dets += n

    if processed % 100 == 0 or n > 0:
        t_sec = (frame_idx / fps) if fps > 0 else 0
        print(f"  [{processed + 1}/{total}] Frame {frame_idx} | t={t_sec:.1f}s | {n} masks")

    processed += 1
    frame_idx += 1

cap.release()
writer.release()

print(f"\n{'=' * 60}")
print(f"  DONE!")
print(f"{'=' * 60}")
print(f"  Frames processed:  {processed}")
print(f"  Total detections:  {total_dets}")
print(f"  Output video:      {OUTPUT_VIDEO}")
print("=" * 60)

[INFO] Loading SAM3: facebook/sam3 | device: cuda


Loading weights:   0%|          | 0/1468 [00:00<?, ?it/s]

[INFO] SAM3 loaded!
  SAM3 VIDEO MASK → VIDEO OUTPUT
  Input:       G:\2026-Research-Project\admision_test\2026-04-15 11-00-45.mp4
  Resolution:  1280x720 @ 30.0 FPS
  Total:       6023 frames
  Prompt:      'vehicle'
  Threshold:   0.5
  Device:      cuda
  Mask color:  Blue (BGR (255, 0, 0))
  Output:      G:\2026-Research-Project\admision_test/output_sam3_masks\segmented_output.mp4
  [1/6023] Frame 0 | t=0.0s | 24 masks
  [2/6023] Frame 1 | t=0.0s | 23 masks
  [3/6023] Frame 2 | t=0.1s | 25 masks
  [4/6023] Frame 3 | t=0.1s | 26 masks
  [5/6023] Frame 4 | t=0.1s | 25 masks
  [6/6023] Frame 5 | t=0.2s | 28 masks
  [7/6023] Frame 6 | t=0.2s | 26 masks
  [8/6023] Frame 7 | t=0.2s | 27 masks
  [9/6023] Frame 8 | t=0.3s | 28 masks
  [10/6023] Frame 9 | t=0.3s | 26 masks
  [11/6023] Frame 10 | t=0.3s | 27 masks
  [12/6023] Frame 11 | t=0.4s | 27 masks
  [13/6023] Frame 12 | t=0.4s | 25 masks
  [14/6023] Frame 13 | t=0.4s | 25 masks
  [15/6023] Frame 14 | t=0.5s | 25 masks
  [16/6023] Fram

In [ ]:
import os
import json
import base64
import time
import cv2
import requests


# =========================
# CONFIG
# =========================

VIDEO_PATH       = r"G:\2026-Research-Project\admision_test\output_sam3_masks\segmented_output.mp4"
FRAME_STRIDE     = 200
START_FRAME      = 0
MAX_FRAMES       = -1

# --- vLLM Server (OpenAI-compatible API) ---
VLLM_BASE_URL    = "http://localhost:8000/v1"
VLLM_CHAT_URL    = f"{VLLM_BASE_URL}/chat/completions"
VLLM_MODELS_URL  = f"{VLLM_BASE_URL}/models"
VLLM_API_KEY     = "EMPTY"              # vLLM không cần API key
LM_MAX_TOKENS    = 2048
# LM_TEMPERATURE   = 0.7
# LM_TOP_P         = 0.95
# LM_TOP_K         = 64                   # Gemma 4 recommended

# --- Prompt: 3 output rõ ràng ---
SYSTEM_PROMPT = """You are an expert traffic surveillance analyst AI.

IMPORTANT CONTEXT: 
- This image is from a traffic camera that has been pre-processed with SAM3 instance segmentation.
- All detected vehicles are highlighted with BLUE OVERLAY MASKS on the image.
- The blue-colored regions represent segmented vehicles (cars, motorcycles, trucks, buses, scooters).
- Use these blue masks to accurately count and classify vehicles.

Your task is to analyze the traffic scene and provide EXACTLY 3 sections of analysis.

Respond ONLY with a valid JSON object using this EXACT structure (no markdown, no extra text, no ```json wrapper):

{
  "traffic_description": "<Comprehensive description of the traffic scene: total vehicle count based on blue masks, types of vehicles (cars, motorcycles, trucks, buses), their positions on the road, movement patterns, lane usage, and overall traffic flow direction. Be specific about numbers.>",
  
  "traffic_density_description": "<Rate the density as one of: Low / Medium / High / Congested. Then explain WHY: describe the spacing between vehicles, road capacity utilization percentage estimate, queue lengths, and compare to typical urban traffic patterns. Mention if traffic is free-flowing, slow-moving, stop-and-go, or gridlocked.>",
  
  "traffic_abnormal_incident_description": "<Describe ANY abnormal incidents or safety concerns observed: accidents, stalled vehicles, wrong-way driving, illegal parking, pedestrians in roadway, debris/obstacles, unusual congestion patterns, emergency vehicles, traffic signal violations. If NO abnormal incidents are detected, explicitly state 'No abnormal incidents detected' and briefly note the general safety condition.>"
}"""

USER_PROMPT = """Analyze this segmented traffic camera frame.
The BLUE highlighted areas are vehicles detected by SAM3 segmentation.
Count the blue-masked vehicles accurately and provide the 3-section analysis in JSON only.
Do NOT wrap your response in ```json``` markers."""

# --- Output ---
OUTPUT_DIR       = "./output_gemma4_traffic"
OUTPUT_JSON      = os.path.join(OUTPUT_DIR, "traffic_analysis.json")
FRAMES_DIR       = os.path.join(OUTPUT_DIR, "analyzed_frames")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)


# =========================
# HELPERS
# =========================
def frame_to_base64(frame_bgr, quality=85):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, buffer = cv2.imencode('.jpg', frame_bgr, encode_param)
    return base64.b64encode(buffer).decode('utf-8')


def query_vllm(base64_img, frame_info=""):
    """Gửi ảnh base64 lên vLLM server để reasoning."""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {VLLM_API_KEY}"
    }

    user_content = [
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{base64_img}"}
        },
        {
            "type": "text",
            "text": f"{USER_PROMPT}\n\nFrame info: {frame_info}"
        }
    ]

    payload = {
        "model": "",   # vLLM tự dùng model đang serve
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        "max_tokens": LM_MAX_TOKENS,
        # "temperature": LM_TEMPERATURE,
        # "top_p": LM_TOP_P,
        # "top_k": LM_TOP_K,
        "stream": False
    }

    try:
        resp = requests.post(VLLM_CHAT_URL, json=payload, headers=headers, timeout=180)
        resp.raise_for_status()
        data = resp.json()

        # Lấy model name từ response
        model_name = data.get("model", "unknown")
        answer = data["choices"][0]["message"]["content"]

        # Parse JSON
        try:
            clean = answer.strip()
            # Strip markdown wrapper nếu có
            if clean.startswith("```"):
                clean = clean.split("\n", 1)[1] if "\n" in clean else clean[3:]
                if clean.endswith("```"):
                    clean = clean[:-3]
                clean = clean.strip()
            parsed = json.loads(clean)
            return {"raw": answer, "parsed": parsed, "error": None, "model": model_name}
        except json.JSONDecodeError:
            # Thử thêm closing braces cho truncated JSON
            for suffix in ["}", "\"}", "\"]}", "\"]}}"]:
                for n in range(1, 5):
                    try:
                        parsed = json.loads(clean + suffix * n)
                        return {"raw": answer, "parsed": parsed, "error": None, "model": model_name}
                    except json.JSONDecodeError:
                        continue
            return {"raw": answer, "parsed": None, "error": None, "model": model_name}

    except requests.exceptions.ConnectionError:
        return {"raw": None, "parsed": None, "error": "Cannot connect to vLLM server. Is it running?", "model": None}
    except requests.exceptions.Timeout:
        return {"raw": None, "parsed": None, "error": "vLLM timeout (>180s)", "model": None}
    except Exception as e:
        return {"raw": None, "parsed": None, "error": str(e), "model": None}


def detect_model():
    """Auto-detect model name từ vLLM server."""
    try:
        headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}
        resp = requests.get(VLLM_MODELS_URL, headers=headers, timeout=10)
        if resp.status_code == 200:
            models = resp.json().get("data", [])
            if models:
                return models[0]["id"]
    except:
        pass
    return None


# =========================
# RUN
# =========================
if not os.path.isfile(VIDEO_PATH):
    raise RuntimeError(f"Video not found: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
W_vid = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_vid = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("=" * 65)
print("  GEMMA 4 TRAFFIC ANALYSIS via vLLM (SEGMENTED VIDEO)")
print("=" * 65)
print(f"  Video:       {VIDEO_PATH}")
print(f"  Resolution:  {W_vid}x{H_vid} @ {fps:.1f} FPS")
print(f"  Total:       {total} frames")
print(f"  Stride:      every {FRAME_STRIDE} frames")
print(f"  ~Frames:     {total // FRAME_STRIDE}")
print(f"  vLLM URL:    {VLLM_BASE_URL}")
print(f"  Max tokens:  {LM_MAX_TOKENS}")
print(f"  Output:      {OUTPUT_DIR}")
print(f"  NOTE:        Blue masks = SAM3 segmented vehicles")
print(f"  OUTPUT:      3 fields: traffic_description,")
print(f"               traffic_density_description,")
print(f"               traffic_abnormal_incident_description")
print("=" * 65)

# Test connection + detect model
print("\n[INFO] Connecting to vLLM server...")
model_name = detect_model()
if model_name:
    print(f"[INFO] Connected! Model: {model_name}")
else:
    print("[WARN] Cannot detect model from vLLM server.")
    print("[WARN] Make sure vLLM is running:")
    print("       vllm serve google/gemma-4-E4B-it --dtype auto --max-model-len 4096 --trust-remote-code --port 8000")
    model_name = "google/gemma-4-E4B-it"

if START_FRAME > 0:
    cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

results = []
frame_idx = START_FRAME
analyzed = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if (frame_idx - START_FRAME) % FRAME_STRIDE != 0:
        frame_idx += 1
        continue

    if 0 < MAX_FRAMES <= analyzed:
        break

    t_sec = (frame_idx / fps) if fps > 0 else 0.0
    frame_info = f"Frame {frame_idx}/{total}, Time {t_sec:.1f}s, Segmented (blue masks = vehicles)"

    print(f"\n[{analyzed + 1}] Frame {frame_idx} | t={t_sec:.1f}s")

    # Save frame
    frame_filename = f"frame_{frame_idx:06d}.jpg"
    cv2.imwrite(os.path.join(FRAMES_DIR, frame_filename), frame)

    # Query vLLM
    print(f"  Querying vLLM...")
    t0 = time.time()
    b64 = frame_to_base64(frame)
    result = query_vllm(b64, frame_info)
    elapsed = time.time() - t0

    # Update model name nếu detect được
    if result["model"]:
        model_name = result["model"]

    entry = {
        "frame_index": frame_idx,
        "timestamp_sec": round(t_sec, 2),
        "frame_file": frame_filename,
        "inference_time_sec": round(elapsed, 2),
        "analysis": result["parsed"] if result["parsed"] else result["raw"],
        "error": result["error"]
    }
    results.append(entry)

    # Log
    if result["error"]:
        print(f"  ERROR: {result['error']}")
    else:
        print(f"  Done in {elapsed:.1f}s")
        if result["parsed"]:
            p = result["parsed"]
            # Preview
            desc = p.get("traffic_description", "")[:80]
            density = p.get("traffic_density_description", "")[:50]
            incident = p.get("traffic_abnormal_incident_description", "")[:50]
            print(f"  Traffic:  {desc}...")
            print(f"  Density:  {density}...")
            print(f"  Incident: {incident}...")
        else:
            print(f"  Raw: {(result['raw'] or '')[:120]}...")

    # Save JSON incremental
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump({
            "video": os.path.basename(VIDEO_PATH),
            "fps": fps,
            "total_frames": total,
            "resolution": f"{W_vid}x{H_vid}",
            "frame_stride": FRAME_STRIDE,
            "model": model_name,
            "server": "vLLM",
            "server_url": VLLM_BASE_URL,
            "note": "Input video pre-processed with SAM3 segmentation (blue masks on vehicles)",
            "output_fields": [
                "traffic_description",
                "traffic_density_description",
                "traffic_abnormal_incident_description"
            ],
            "system_prompt": SYSTEM_PROMPT,
            "analyses": results
        }, f, ensure_ascii=False, indent=2)

    analyzed += 1
    frame_idx += 1

cap.release()

print(f"\n{'=' * 65}")
print(f"  DONE!")
print(f"{'=' * 65}")
print(f"  Frames analyzed:   {analyzed}")
print(f"  Model:             {model_name}")
print(f"  Server:            vLLM @ {VLLM_BASE_URL}")
print(f"  Output JSON:       {OUTPUT_JSON}")
print(f"  Analyzed frames:   {FRAMES_DIR}/")
print("=" * 65)

  GEMMA 4 TRAFFIC ANALYSIS via vLLM (SEGMENTED VIDEO)
  Video:       G:\2026-Research-Project\admision_test\output_sam3_masks\segmented_output.mp4
  Resolution:  1280x720 @ 30.0 FPS
  Total:       6023 frames
  Stride:      every 200 frames
  ~Frames:     30
  vLLM URL:    http://localhost:8000/v1
  Max tokens:  2048
  Output:      ./output_gemma4_traffic
  NOTE:        Blue masks = SAM3 segmented vehicles
  OUTPUT:      3 fields: traffic_description,
               traffic_density_description,
               traffic_abnormal_incident_description

[INFO] Connecting to vLLM server...
[INFO] Connected! Model: google/gemma-4-E4B-it

[1] Frame 0 | t=0.0s
  Querying vLLM...
  Done in 15.8s
  Traffic:  There are a total of 17 vehicles detected in the traffic scene based on the blue...
  Density:  Medium. The vehicles are spaced out sufficiently t...
  Incident: No abnormal incidents detected. The traffic appear...

[2] Frame 200 | t=6.7s
  Querying vLLM...
  Done in 8.9s
  Traffic:  There ar

In [ ]:
import os
import re
import json
import textwrap
import cv2
import numpy as np


# =========================
# CONFIG
# =========================

FRAMES_DIR       = "./output_gemma4_traffic/analyzed_frames"
JSON_PATH        = "./output_gemma4_traffic/traffic_analysis.json"
OUTPUT_DIR       = "./output_gemma4_traffic/frames_with_panel"

PANEL_BG         = (30, 30, 30)
MARGIN           = 22
SEPARATOR_W      = 4
SEPARATOR_COLOR  = (80, 80, 80)
VAL_INDENT       = 18
MAX_CHARS_LINE   = 65

COL_TITLE    = (0, 200, 255)      # Vàng cam
COL_KEY      = (120, 220, 120)    # Xanh lá
COL_VAL      = (220, 220, 220)    # Trắng nhạt
COL_HEADER   = (50, 50, 50)
COL_DIVIDER  = (65, 65, 65)
COL_DIM      = (140, 140, 140)
COL_SAFE     = (80, 220, 80)      # Xanh lá cho "No incident"
COL_DANGER   = (60, 60, 255)      # Đỏ cho incident
COL_COUNT    = (255, 255, 255)    # Trắng đậm cho số xe

DENSITY_COL = {
    "low":       (0, 255, 0),
    "medium":    (0, 230, 230),
    "high":      (0, 165, 255),
    "congested": (0, 0, 255),
}

FONT       = cv2.FONT_HERSHEY_SIMPLEX
FONT_AA    = cv2.LINE_AA
F_SCALE    = 0.52
LINE_H     = 25
KEY_H      = 32
SECTION_GAP = 14

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# HELPERS
# =========================
def put(img, text, x, y, scale, color, thick=1):
    cv2.putText(img, str(text), (x, y), FONT, scale, color, thick, FONT_AA)

def text_w(text, scale=F_SCALE, thick=1):
    return cv2.getTextSize(str(text), FONT, scale, thick)[0][0]

def hline(img, x1, x2, y, color=COL_DIVIDER, thick=1):
    cv2.line(img, (x1, y), (x2, y), color, thick, FONT_AA)

def wrap(text, max_chars=MAX_CHARS_LINE):
    text = str(text).replace("\n", " ").strip()
    return textwrap.wrap(text, width=max_chars) or [""]


def extract_vehicle_count(desc):
    """Trích số xe từ traffic_description."""
    m = re.search(r'(?:total of |are |are a total of )(\d+)', desc, re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r'(\d+)\s*(?:vehicles|blue.?masked)', desc, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return None


def extract_density_rating(density_desc):
    """Trích rating keyword từ đầu density description."""
    text = density_desc.strip()
    # Format: "Medium. The vehicles are..."
    m = re.match(r'^(Low|Medium|High|Congested)[\.\s]', text, re.IGNORECASE)
    if m:
        return m.group(1)
    # Format: "Medium-High. ..."
    m = re.match(r'^([\w\-/]+)[\.\s]', text)
    if m:
        return m.group(1)
    return None


def density_color(rating):
    if not rating:
        return COL_VAL
    r = rating.lower()
    for k, c in DENSITY_COL.items():
        if k in r:
            return c
    return COL_VAL


def is_safe(incident_desc):
    return "no abnormal" in incident_desc.lower()


# =========================
# CALC PANEL WIDTH
# =========================
def calc_panel_width(analysis):
    max_w = text_w("TRAFFIC ANALYSIS", 0.7)
    labels = ["Traffic Description:", "Traffic Density:", "Abnormal Incidents:"]
    for label in labels:
        max_w = max(max_w, text_w(label, F_SCALE))

    for field in ["traffic_description", "traffic_density_description",
                   "traffic_abnormal_incident_description"]:
        val = analysis.get(field, "")
        lines = wrap(val)
        for line in lines:
            max_w = max(max_w, text_w(line, F_SCALE) + VAL_INDENT)

    return int(max_w + MARGIN * 2 + 20)


# =========================
# BUILD PANEL
# =========================
def build_panel(frame_h, entry, panel_w):
    analysis = entry.get("analysis", {})
    if not isinstance(analysis, dict):
        analysis = {}

    panel = np.full((frame_h, panel_w, 3), PANEL_BG, dtype=np.uint8)
    m = MARGIN
    rw = panel_w - 2 * m

    # ── HEADER ──
    header_h = 80
    cv2.rectangle(panel, (0, 0), (panel_w, header_h), COL_HEADER, -1)

    put(panel, "TRAFFIC ANALYSIS", m, 26, 0.7, COL_TITLE, 2)

    # Frame info
    fidx = entry.get("frame_index", "?")
    tsec = entry.get("timestamp_sec", 0)
    inft = entry.get("inference_time_sec", 0)
    put(panel, f"Frame {fidx}  |  t = {tsec:.1f}s  |  {inft:.1f}s",
        m, 48, 0.38, COL_DIM)

    # Vehicle count — nổi bật bên phải header
    desc = analysis.get("traffic_description", "")
    vcount = extract_vehicle_count(desc)
    if vcount is not None:
        count_text = str(vcount)
        label_text = "vehicles"
        cw = text_w(count_text, 0.9, 2) + text_w(label_text, 0.4) + 8
        cx = panel_w - m - cw
        put(panel, count_text, cx, 30, 0.9, COL_COUNT, 2)
        put(panel, label_text, cx + text_w(count_text, 0.9, 2) + 8, 30, 0.4, COL_DIM)

    # Density rating — badge bên phải
    density_desc = analysis.get("traffic_density_description", "")
    rating = extract_density_rating(density_desc)
    if rating:
        rcol = density_color(rating)
        put(panel, rating.upper(), panel_w - m - text_w(rating.upper(), 0.5, 2), 56, 0.5, rcol, 2)

    # Incident status indicator
    incident_desc = analysis.get("traffic_abnormal_incident_description", "")
    safe = is_safe(incident_desc)
    status_text = "SAFE" if safe else "ALERT"
    status_col = COL_SAFE if safe else COL_DANGER
    sw = text_w(status_text, 0.38, 1)
    put(panel, status_text, panel_w - m - sw, 72, 0.38, status_col, 1)

    y = header_h + 12

    # ── SECTIONS ──
    sections = [
        ("Traffic Description:",  "traffic_description",  COL_VAL),
        ("Traffic Density:",      "traffic_density_description", None),  # color handled separately
        ("Abnormal Incidents:",   "traffic_abnormal_incident_description", None),
    ]

    for label, field, default_col in sections:
        if y + KEY_H > frame_h - 10:
            break

        val = analysis.get(field, "")
        if not val:
            continue

        # Key (xanh lá)
        put(panel, label, m, y + 20, F_SCALE, COL_KEY, 1)
        y += KEY_H

        # Determine color
        if field == "traffic_density_description":
            # Rating word in color, rest in white
            rating_word = extract_density_rating(val)
            if rating_word:
                rcol = density_color(rating_word)
                # First line: rating in color
                put(panel, rating_word + ".", m + VAL_INDENT, y + 5, F_SCALE, rcol, 1)
                # Rest of text after rating
                rest = val[len(rating_word) + 1:].strip()
                lines = wrap(rest)
                y += LINE_H
                for line in lines:
                    if y + LINE_H > frame_h - 10:
                        break
                    put(panel, line, m + VAL_INDENT, y + 5, F_SCALE, COL_VAL, 1)
                    y += LINE_H
            else:
                lines = wrap(val)
                for line in lines:
                    if y + LINE_H > frame_h - 10:
                        break
                    put(panel, line, m + VAL_INDENT, y + 5, F_SCALE, COL_VAL, 1)
                    y += LINE_H

        elif field == "traffic_abnormal_incident_description":
            safe = is_safe(val)
            vcol = COL_SAFE if safe else COL_DANGER
            lines = wrap(val)
            for line in lines:
                if y + LINE_H > frame_h - 10:
                    break
                put(panel, line, m + VAL_INDENT, y + 5, F_SCALE, vcol, 1)
                y += LINE_H

        else:
            lines = wrap(val)
            for line in lines:
                if y + LINE_H > frame_h - 10:
                    break
                put(panel, line, m + VAL_INDENT, y + 5, F_SCALE, default_col or COL_VAL, 1)
                y += LINE_H

        y += 4
        hline(panel, m, m + rw, y, COL_DIVIDER)
        y += SECTION_GAP

    return panel


# =========================
# MAIN
# =========================
def main():
    print(f"[INFO] Loading JSON: {JSON_PATH}")
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    analyses = data.get("analyses", [])
    print(f"[INFO] {len(analyses)} entries")
    print("=" * 60)

    processed = 0
    for i, entry in enumerate(analyses):
        frame_file = entry.get("frame_file", "")
        frame_idx = entry.get("frame_index", 0)

        candidates = [
            os.path.join(FRAMES_DIR, frame_file),
            os.path.join(FRAMES_DIR, f"frame_{frame_idx:06d}.jpg"),
        ]

        img = None
        for path in candidates:
            if os.path.isfile(path):
                img = cv2.imread(path)
                break

        if img is None:
            print(f"  [{i+1:2d}] SKIP - no image for frame {frame_idx}")
            continue

        H, W = img.shape[:2]

        # Panel width
        analysis = entry.get("analysis", {})
        if isinstance(analysis, dict):
            panel_w = calc_panel_width(analysis)
        else:
            panel_w = 600
        panel_w = max(500, min(panel_w, 1000))

        # Build panel
        panel = build_panel(H, entry, panel_w)

        # Combine
        total_w = W + SEPARATOR_W + panel_w
        combined = np.full((H, total_w, 3), PANEL_BG, dtype=np.uint8)
        combined[:H, :W] = img
        combined[:, W:W+SEPARATOR_W] = SEPARATOR_COLOR
        combined[:H, W+SEPARATOR_W:] = panel

        out_name = f"frame_{frame_idx:06d}_panel.jpg"
        cv2.imwrite(os.path.join(OUTPUT_DIR, out_name), combined)

        # Log
        if isinstance(analysis, dict):
            desc = analysis.get("traffic_description", "")
            vcount = extract_vehicle_count(desc)
            rating = extract_density_rating(analysis.get("traffic_density_description", ""))
            safe = is_safe(analysis.get("traffic_abnormal_incident_description", ""))
            status = "SAFE" if safe else "ALERT"
            print(f"  [{i+1:2d}] Frame {frame_idx:5d} | {vcount or '?':>3} vehicles | {rating or '?':10s} | {status}")
        else:
            print(f"  [{i+1:2d}] Frame {frame_idx:5d} | no data")

        processed += 1

    print(f"\n  DONE! {processed} images -> {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

[INFO] Loading JSON: ./output_gemma4_traffic/traffic_analysis.json
[INFO] 31 entries
  [ 1] Frame     0 |  17 vehicles | Medium     | SAFE
  [ 2] Frame   200 |  16 vehicles | Medium     | SAFE
  [ 3] Frame   400 |  16 vehicles | Medium     | SAFE
  [ 4] Frame   600 |  15 vehicles | Medium     | SAFE
  [ 5] Frame   800 |  16 vehicles | Medium     | SAFE
  [ 6] Frame  1000 |  18 vehicles | Medium     | SAFE
  [ 7] Frame  1200 |  19 vehicles | Medium     | SAFE
  [ 8] Frame  1400 |  12 vehicles | Medium     | SAFE
  [ 9] Frame  1600 |  16 vehicles | Medium     | SAFE
  [10] Frame  1800 |  16 vehicles | Medium     | SAFE
  [11] Frame  2000 |  21 vehicles | Medium     | SAFE
  [12] Frame  2200 |  14 vehicles | Medium     | SAFE
  [13] Frame  2400 |  16 vehicles | Medium     | SAFE
  [14] Frame  2600 |  15 vehicles | Medium     | SAFE
  [15] Frame  2800 |  14 vehicles | Medium     | SAFE
  [16] Frame  3000 |  17 vehicles | Medium     | SAFE
  [17] Frame  3200 |  19 vehicles | Medium     | SA

#VLM integration in case of non-segmented vehicles frames

In [ ]:
import os
import json
import base64
import time
import cv2
import requests


# =========================
# CONFIG
# =========================

VIDEO_PATH       = r"G:\2026-Research-Project\admision_test\2026-04-15 11-00-45.mp4"
FRAME_STRIDE     = 200
START_FRAME      = 0
MAX_FRAMES       = -1

# --- vLLM Server (OpenAI-compatible API) ---
VLLM_BASE_URL    = "http://localhost:8000/v1"
VLLM_CHAT_URL    = f"{VLLM_BASE_URL}/chat/completions"
VLLM_MODELS_URL  = f"{VLLM_BASE_URL}/models"
VLLM_API_KEY     = "EMPTY"              # vLLM không cần API key
LM_MAX_TOKENS    = 2048
# LM_TEMPERATURE   = 0.7
# LM_TOP_P         = 0.95
# LM_TOP_K         = 64                   # Gemma 4 recommended

# --- Prompt: 3 output rõ ràng ---
SYSTEM_PROMPT = """You are an expert traffic surveillance analyst AI.

IMPORTANT CONTEXT: 
- This image is from a traffic camera that has been pre-processed with SAM3 instance segmentation.

Your task is to analyze the traffic scene and provide EXACTLY 3 sections of analysis.

Respond ONLY with a valid JSON object using this EXACT structure (no markdown, no extra text, no ```json wrapper):

{
  "traffic_description": "<Comprehensive description of the traffic scene: total vehicle count, types of vehicles (cars, motorcycles, trucks, buses), their positions on the road, movement patterns, lane usage, and overall traffic flow direction. Be specific about numbers.>",
  
  "traffic_density_description": "<Rate the density as one of: Low / Medium / High / Congested. Then explain WHY: describe the spacing between vehicles, road capacity utilization percentage estimate, queue lengths, and compare to typical urban traffic patterns. Mention if traffic is free-flowing, slow-moving, stop-and-go, or gridlocked.>",
  
  "traffic_abnormal_incident_description": "<Describe ANY abnormal incidents or safety concerns observed: accidents, stalled vehicles, wrong-way driving, illegal parking, pedestrians in roadway, debris/obstacles, unusual congestion patterns, emergency vehicles, traffic signal violations. If NO abnormal incidents are detected, explicitly state 'No abnormal incidents detected' and briefly note the general safety condition.>"
}"""

USER_PROMPT = """Analyze this segmented traffic camera frame.
Count the vehicles accurately and provide the 3-section analysis in JSON only.
Do NOT wrap your response in ```json``` markers."""

# --- Output ---
OUTPUT_DIR       = "./output_gemma4_traffic_no_segment_v2"
OUTPUT_JSON      = os.path.join(OUTPUT_DIR, "traffic_analysis.json")
FRAMES_DIR       = os.path.join(OUTPUT_DIR, "analyzed_frames")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FRAMES_DIR, exist_ok=True)


# =========================
# HELPERS
# =========================
def frame_to_base64(frame_bgr, quality=85):
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), quality]
    _, buffer = cv2.imencode('.jpg', frame_bgr, encode_param)
    return base64.b64encode(buffer).decode('utf-8')


def query_vllm(base64_img, frame_info=""):
    """Gửi ảnh base64 lên vLLM server để reasoning."""
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {VLLM_API_KEY}"
    }

    user_content = [
        {
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{base64_img}"}
        },
        {
            "type": "text",
            "text": f"{USER_PROMPT}\n\nFrame info: {frame_info}"
        }
    ]

    payload = {
        "model": "",   # vLLM tự dùng model đang serve
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}
        ],
        "max_tokens": LM_MAX_TOKENS,
        # "temperature": LM_TEMPERATURE,
        # "top_p": LM_TOP_P,
        # "top_k": LM_TOP_K,
        "stream": False
    }

    try:
        resp = requests.post(VLLM_CHAT_URL, json=payload, headers=headers, timeout=180)
        resp.raise_for_status()
        data = resp.json()

        # Lấy model name từ response
        model_name = data.get("model", "unknown")
        answer = data["choices"][0]["message"]["content"]

        # Parse JSON
        try:
            clean = answer.strip()
            # Strip markdown wrapper nếu có
            if clean.startswith("```"):
                clean = clean.split("\n", 1)[1] if "\n" in clean else clean[3:]
                if clean.endswith("```"):
                    clean = clean[:-3]
                clean = clean.strip()
            parsed = json.loads(clean)
            return {"raw": answer, "parsed": parsed, "error": None, "model": model_name}
        except json.JSONDecodeError:
            # Thử thêm closing braces cho truncated JSON
            for suffix in ["}", "\"}", "\"]}", "\"]}}"]:
                for n in range(1, 5):
                    try:
                        parsed = json.loads(clean + suffix * n)
                        return {"raw": answer, "parsed": parsed, "error": None, "model": model_name}
                    except json.JSONDecodeError:
                        continue
            return {"raw": answer, "parsed": None, "error": None, "model": model_name}

    except requests.exceptions.ConnectionError:
        return {"raw": None, "parsed": None, "error": "Cannot connect to vLLM server. Is it running?", "model": None}
    except requests.exceptions.Timeout:
        return {"raw": None, "parsed": None, "error": "vLLM timeout (>180s)", "model": None}
    except Exception as e:
        return {"raw": None, "parsed": None, "error": str(e), "model": None}


def detect_model():
    """Auto-detect model name từ vLLM server."""
    try:
        headers = {"Authorization": f"Bearer {VLLM_API_KEY}"}
        resp = requests.get(VLLM_MODELS_URL, headers=headers, timeout=10)
        if resp.status_code == 200:
            models = resp.json().get("data", [])
            if models:
                return models[0]["id"]
    except:
        pass
    return None


# =========================
# RUN
# =========================
if not os.path.isfile(VIDEO_PATH):
    raise RuntimeError(f"Video not found: {VIDEO_PATH}")

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    raise RuntimeError(f"Cannot open video: {VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
W_vid = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H_vid = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("=" * 65)
print("  GEMMA 4 TRAFFIC ANALYSIS via vLLM (SEGMENTED VIDEO)")
print("=" * 65)
print(f"  Video:       {VIDEO_PATH}")
print(f"  Resolution:  {W_vid}x{H_vid} @ {fps:.1f} FPS")
print(f"  Total:       {total} frames")
print(f"  Stride:      every {FRAME_STRIDE} frames")
print(f"  ~Frames:     {total // FRAME_STRIDE}")
print(f"  vLLM URL:    {VLLM_BASE_URL}")
print(f"  Max tokens:  {LM_MAX_TOKENS}")
print(f"  Output:      {OUTPUT_DIR}")
print(f"  NOTE:        Blue masks = SAM3 segmented vehicles")
print(f"  OUTPUT:      3 fields: traffic_description,")
print(f"               traffic_density_description,")
print(f"               traffic_abnormal_incident_description")
print("=" * 65)

# Test connection + detect model
print("\n[INFO] Connecting to vLLM server...")
model_name = detect_model()
if model_name:
    print(f"[INFO] Connected! Model: {model_name}")
else:
    print("[WARN] Cannot detect model from vLLM server.")
    print("[WARN] Make sure vLLM is running:")
    print("       vllm serve google/gemma-4-E4B-it --dtype auto --max-model-len 4096 --trust-remote-code --port 8000")
    model_name = "google/gemma-4-E4B-it"

if START_FRAME > 0:
    cap.set(cv2.CAP_PROP_POS_FRAMES, START_FRAME)

results = []
frame_idx = START_FRAME
analyzed = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break

    if (frame_idx - START_FRAME) % FRAME_STRIDE != 0:
        frame_idx += 1
        continue

    if 0 < MAX_FRAMES <= analyzed:
        break

    t_sec = (frame_idx / fps) if fps > 0 else 0.0
    frame_info = f"Frame {frame_idx}/{total}, Time {t_sec:.1f}s, Segmented (blue masks = vehicles)"

    print(f"\n[{analyzed + 1}] Frame {frame_idx} | t={t_sec:.1f}s")

    # Save frame
    frame_filename = f"frame_{frame_idx:06d}.jpg"
    cv2.imwrite(os.path.join(FRAMES_DIR, frame_filename), frame)

    # Query vLLM
    print(f"  Querying vLLM...")
    t0 = time.time()
    b64 = frame_to_base64(frame)
    result = query_vllm(b64, frame_info)
    elapsed = time.time() - t0

    # Update model name nếu detect được
    if result["model"]:
        model_name = result["model"]

    entry = {
        "frame_index": frame_idx,
        "timestamp_sec": round(t_sec, 2),
        "frame_file": frame_filename,
        "inference_time_sec": round(elapsed, 2),
        "analysis": result["parsed"] if result["parsed"] else result["raw"],
        "error": result["error"]
    }
    results.append(entry)

    # Log
    if result["error"]:
        print(f"  ERROR: {result['error']}")
    else:
        print(f"  Done in {elapsed:.1f}s")
        if result["parsed"]:
            p = result["parsed"]
            # Preview
            desc = p.get("traffic_description", "")[:80]
            density = p.get("traffic_density_description", "")[:50]
            incident = p.get("traffic_abnormal_incident_description", "")[:50]
            print(f"  Traffic:  {desc}...")
            print(f"  Density:  {density}...")
            print(f"  Incident: {incident}...")
        else:
            print(f"  Raw: {(result['raw'] or '')[:120]}...")

    # Save JSON incremental
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump({
            "video": os.path.basename(VIDEO_PATH),
            "fps": fps,
            "total_frames": total,
            "resolution": f"{W_vid}x{H_vid}",
            "frame_stride": FRAME_STRIDE,
            "model": model_name,
            "server": "vLLM",
            "server_url": VLLM_BASE_URL,
            "note": "Input video pre-processed with SAM3 segmentation (blue masks on vehicles)",
            "output_fields": [
                "traffic_description",
                "traffic_density_description",
                "traffic_abnormal_incident_description"
            ],
            "system_prompt": SYSTEM_PROMPT,
            "analyses": results
        }, f, ensure_ascii=False, indent=2)

    analyzed += 1
    frame_idx += 1

cap.release()

print(f"\n{'=' * 65}")
print(f"  DONE!")
print(f"{'=' * 65}")
print(f"  Frames analyzed:   {analyzed}")
print(f"  Model:             {model_name}")
print(f"  Server:            vLLM @ {VLLM_BASE_URL}")
print(f"  Output JSON:       {OUTPUT_JSON}")
print(f"  Analyzed frames:   {FRAMES_DIR}/")
print("=" * 65)

  GEMMA 4 TRAFFIC ANALYSIS via vLLM (SEGMENTED VIDEO)
  Video:       G:\2026-Research-Project\admision_test\2026-04-15 11-00-45.mp4
  Resolution:  1280x720 @ 30.0 FPS
  Total:       6023 frames
  Stride:      every 200 frames
  ~Frames:     30
  vLLM URL:    http://localhost:8000/v1
  Max tokens:  2048
  Output:      ./output_gemma4_traffic_no_segment_v2
  NOTE:        Blue masks = SAM3 segmented vehicles
  OUTPUT:      3 fields: traffic_description,
               traffic_density_description,
               traffic_abnormal_incident_description

[INFO] Connecting to vLLM server...
[INFO] Connected! Model: google/gemma-4-E4B-it

[1] Frame 0 | t=0.0s
  Querying vLLM...
  Done in 7.7s
  Traffic:  The scene shows a multi-lane urban road with moderate traffic flow. Vehicles are...
  Density:  Medium. The vehicles are spaced reasonably apart, ...
  Incident: No abnormal incidents detected. The general safety...

[2] Frame 200 | t=6.7s
  Querying vLLM...
  Done in 9.1s
  Traffic:  The scene 

In [ ]:
"""
Traffic Analysis Panel Renderer (vLLM v3 output)
===================================================
3 sections: Traffic Description, Density, Abnormal Incidents
- Vehicle count trích từ description → hiển thị nổi bật ở header
- Density rating có màu (xanh/vàng/cam/đỏ)
- Incident status: xanh nếu safe, đỏ nếu có sự cố
"""

import os
import re
import json
import textwrap
import cv2
import numpy as np


# =========================
# CONFIG
# =========================

FRAMES_DIR       = "./output_gemma4_traffic_no_segment_v2/analyzed_frames"
JSON_PATH        = "./output_gemma4_traffic_no_segment_v2/traffic_analysis.json"
OUTPUT_DIR       = "./output_gemma4_traffic_no_segment_v2/frames_with_panel"

PANEL_BG         = (30, 30, 30)
MARGIN           = 22
SEPARATOR_W      = 4
SEPARATOR_COLOR  = (80, 80, 80)
VAL_INDENT       = 18
MAX_CHARS_LINE   = 65

COL_TITLE    = (0, 200, 255)      # Vàng cam
COL_KEY      = (120, 220, 120)    # Xanh lá
COL_VAL      = (220, 220, 220)    # Trắng nhạt
COL_HEADER   = (50, 50, 50)
COL_DIVIDER  = (65, 65, 65)
COL_DIM      = (140, 140, 140)
COL_SAFE     = (80, 220, 80)      # Xanh lá cho "No incident"
COL_DANGER   = (60, 60, 255)      # Đỏ cho incident
COL_COUNT    = (255, 255, 255)    # Trắng đậm cho số xe

DENSITY_COL = {
    "low":       (0, 255, 0),
    "medium":    (0, 230, 230),
    "high":      (0, 165, 255),
    "congested": (0, 0, 255),
}

FONT       = cv2.FONT_HERSHEY_SIMPLEX
FONT_AA    = cv2.LINE_AA
F_SCALE    = 0.52
LINE_H     = 25
KEY_H      = 32
SECTION_GAP = 14

os.makedirs(OUTPUT_DIR, exist_ok=True)


# =========================
# HELPERS
# =========================
def put(img, text, x, y, scale, color, thick=1):
    cv2.putText(img, str(text), (x, y), FONT, scale, color, thick, FONT_AA)

def text_w(text, scale=F_SCALE, thick=1):
    return cv2.getTextSize(str(text), FONT, scale, thick)[0][0]

def hline(img, x1, x2, y, color=COL_DIVIDER, thick=1):
    cv2.line(img, (x1, y), (x2, y), color, thick, FONT_AA)

def wrap(text, max_chars=MAX_CHARS_LINE):
    text = str(text).replace("\n", " ").strip()
    return textwrap.wrap(text, width=max_chars) or [""]


def extract_vehicle_count(desc):
    """Trích số xe từ traffic_description."""
    m = re.search(r'(?:total of |are |are a total of )(\d+)', desc, re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r'(\d+)\s*(?:vehicles|blue.?masked)', desc, re.IGNORECASE)
    if m:
        return int(m.group(1))
    return None


def extract_density_rating(density_desc):
    """Trích rating keyword từ đầu density description."""
    text = density_desc.strip()
    # Format: "Medium. The vehicles are..."
    m = re.match(r'^(Low|Medium|High|Congested)[\.\s]', text, re.IGNORECASE)
    if m:
        return m.group(1)
    # Format: "Medium-High. ..."
    m = re.match(r'^([\w\-/]+)[\.\s]', text)
    if m:
        return m.group(1)
    return None


def density_color(rating):
    if not rating:
        return COL_VAL
    r = rating.lower()
    for k, c in DENSITY_COL.items():
        if k in r:
            return c
    return COL_VAL


def is_safe(incident_desc):
    return "no abnormal" in incident_desc.lower()


# =========================
# CALC PANEL WIDTH
# =========================
def calc_panel_width(analysis):
    max_w = text_w("TRAFFIC ANALYSIS", 0.7)
    labels = ["Traffic Description:", "Traffic Density:", "Abnormal Incidents:"]
    for label in labels:
        max_w = max(max_w, text_w(label, F_SCALE))

    for field in ["traffic_description", "traffic_density_description",
                   "traffic_abnormal_incident_description"]:
        val = analysis.get(field, "")
        lines = wrap(val)
        for line in lines:
            max_w = max(max_w, text_w(line, F_SCALE) + VAL_INDENT)

    return int(max_w + MARGIN * 2 + 20)


# =========================
# BUILD PANEL
# =========================
def build_panel(frame_h, entry, panel_w):
    analysis = entry.get("analysis", {})
    if not isinstance(analysis, dict):
        analysis = {}

    panel = np.full((frame_h, panel_w, 3), PANEL_BG, dtype=np.uint8)
    m = MARGIN
    rw = panel_w - 2 * m

    # ── HEADER ──
    header_h = 80
    cv2.rectangle(panel, (0, 0), (panel_w, header_h), COL_HEADER, -1)

    put(panel, "TRAFFIC ANALYSIS", m, 26, 0.7, COL_TITLE, 2)

    # Frame info
    fidx = entry.get("frame_index", "?")
    tsec = entry.get("timestamp_sec", 0)
    inft = entry.get("inference_time_sec", 0)
    put(panel, f"Frame {fidx}  |  t = {tsec:.1f}s  |  {inft:.1f}s",
        m, 48, 0.38, COL_DIM)

    # Vehicle count — nổi bật bên phải header
    desc = analysis.get("traffic_description", "")
    vcount = extract_vehicle_count(desc)
    if vcount is not None:
        count_text = str(vcount)
        label_text = "vehicles"
        cw = text_w(count_text, 0.9, 2) + text_w(label_text, 0.4) + 8
        cx = panel_w - m - cw
        put(panel, count_text, cx, 30, 0.9, COL_COUNT, 2)
        put(panel, label_text, cx + text_w(count_text, 0.9, 2) + 8, 30, 0.4, COL_DIM)

    # Density rating — badge bên phải
    density_desc = analysis.get("traffic_density_description", "")
    rating = extract_density_rating(density_desc)
    if rating:
        rcol = density_color(rating)
        put(panel, rating.upper(), panel_w - m - text_w(rating.upper(), 0.5, 2), 56, 0.5, rcol, 2)

    # Incident status indicator
    incident_desc = analysis.get("traffic_abnormal_incident_description", "")
    safe = is_safe(incident_desc)
    status_text = "SAFE" if safe else "ALERT"
    status_col = COL_SAFE if safe else COL_DANGER
    sw = text_w(status_text, 0.38, 1)
    put(panel, status_text, panel_w - m - sw, 72, 0.38, status_col, 1)

    y = header_h + 12

    # ── SECTIONS ──
    sections = [
        ("Traffic Description:",  "traffic_description",  COL_VAL),
        ("Traffic Density:",      "traffic_density_description", None),  # color handled separately
        ("Abnormal Incidents:",   "traffic_abnormal_incident_description", None),
    ]

    for label, field, default_col in sections:
        if y + KEY_H > frame_h - 10:
            break

        val = analysis.get(field, "")
        if not val:
            continue

        # Key (xanh lá)
        put(panel, label, m, y + 20, F_SCALE, COL_KEY, 1)
        y += KEY_H

        # Determine color
        if field == "traffic_density_description":
            # Rating word in color, rest in white
            rating_word = extract_density_rating(val)
            if rating_word:
                rcol = density_color(rating_word)
                # First line: rating in color
                put(panel, rating_word + ".", m + VAL_INDENT, y + 5, F_SCALE, rcol, 1)
                # Rest of text after rating
                rest = val[len(rating_word) + 1:].strip()
                lines = wrap(rest)
                y += LINE_H
                for line in lines:
                    if y + LINE_H > frame_h - 10:
                        break
                    put(panel, line, m + VAL_INDENT, y + 5, F_SCALE, COL_VAL, 1)
                    y += LINE_H
            else:
                lines = wrap(val)
                for line in lines:
                    if y + LINE_H > frame_h - 10:
                        break
                    put(panel, line, m + VAL_INDENT, y + 5, F_SCALE, COL_VAL, 1)
                    y += LINE_H

        elif field == "traffic_abnormal_incident_description":
            safe = is_safe(val)
            vcol = COL_SAFE if safe else COL_DANGER
            lines = wrap(val)
            for line in lines:
                if y + LINE_H > frame_h - 10:
                    break
                put(panel, line, m + VAL_INDENT, y + 5, F_SCALE, vcol, 1)
                y += LINE_H

        else:
            lines = wrap(val)
            for line in lines:
                if y + LINE_H > frame_h - 10:
                    break
                put(panel, line, m + VAL_INDENT, y + 5, F_SCALE, default_col or COL_VAL, 1)
                y += LINE_H

        y += 4
        hline(panel, m, m + rw, y, COL_DIVIDER)
        y += SECTION_GAP

    return panel


# =========================
# MAIN
# =========================
def main():
    print(f"[INFO] Loading JSON: {JSON_PATH}")
    with open(JSON_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    analyses = data.get("analyses", [])
    print(f"[INFO] {len(analyses)} entries")
    print("=" * 60)

    processed = 0
    for i, entry in enumerate(analyses):
        frame_file = entry.get("frame_file", "")
        frame_idx = entry.get("frame_index", 0)

        candidates = [
            os.path.join(FRAMES_DIR, frame_file),
            os.path.join(FRAMES_DIR, f"frame_{frame_idx:06d}.jpg"),
        ]

        img = None
        for path in candidates:
            if os.path.isfile(path):
                img = cv2.imread(path)
                break

        if img is None:
            print(f"  [{i+1:2d}] SKIP - no image for frame {frame_idx}")
            continue

        H, W = img.shape[:2]

        # Panel width
        analysis = entry.get("analysis", {})
        if isinstance(analysis, dict):
            panel_w = calc_panel_width(analysis)
        else:
            panel_w = 600
        panel_w = max(500, min(panel_w, 1000))

        # Build panel
        panel = build_panel(H, entry, panel_w)

        # Combine
        total_w = W + SEPARATOR_W + panel_w
        combined = np.full((H, total_w, 3), PANEL_BG, dtype=np.uint8)
        combined[:H, :W] = img
        combined[:, W:W+SEPARATOR_W] = SEPARATOR_COLOR
        combined[:H, W+SEPARATOR_W:] = panel

        out_name = f"frame_{frame_idx:06d}_panel.jpg"
        cv2.imwrite(os.path.join(OUTPUT_DIR, out_name), combined)

        # Log
        if isinstance(analysis, dict):
            desc = analysis.get("traffic_description", "")
            vcount = extract_vehicle_count(desc)
            rating = extract_density_rating(analysis.get("traffic_density_description", ""))
            safe = is_safe(analysis.get("traffic_abnormal_incident_description", ""))
            status = "SAFE" if safe else "ALERT"
            print(f"  [{i+1:2d}] Frame {frame_idx:5d} | {vcount or '?':>3} vehicles | {rating or '?':10s} | {status}")
        else:
            print(f"  [{i+1:2d}] Frame {frame_idx:5d} | no data")

        processed += 1

    print(f"\n  DONE! {processed} images -> {OUTPUT_DIR}/")


if __name__ == "__main__":
    main()

[INFO] Loading JSON: ./output_gemma4_traffic_no_segment/traffic_analysis.json
[INFO] 31 entries
  [ 1] Frame     0 |  15 vehicles | Medium     | SAFE
  [ 2] Frame   200 |  18 vehicles | Medium     | SAFE
  [ 3] Frame   400 |  12 vehicles | Medium     | SAFE
  [ 4] Frame   600 |  18 vehicles | Medium     | SAFE
  [ 5] Frame   800 |  21 vehicles | Medium     | SAFE
  [ 6] Frame  1000 |  19 vehicles | Medium     | SAFE
  [ 7] Frame  1200 |  16 vehicles | Medium     | SAFE
  [ 8] Frame  1400 |  13 vehicles | Medium     | SAFE
  [ 9] Frame  1600 |  15 vehicles | Medium     | SAFE
  [10] Frame  1800 |  17 vehicles | Medium     | SAFE
  [11] Frame  2000 |  23 vehicles | Medium     | SAFE
  [12] Frame  2200 |  15 vehicles | Medium     | SAFE
  [13] Frame  2400 |  15 vehicles | Medium     | SAFE
  [14] Frame  2600 |  16 vehicles | Medium     | SAFE
  [15] Frame  2800 |  16 vehicles | Medium     | SAFE
  [16] Frame  3000 |  17 vehicles | Medium     | SAFE
  [17] Frame  3200 |  20 vehicles | Medi